In [1]:
!pip -q install -U transformers sentencepiece gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model loaded!")

Using device: cpu


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded!


In [3]:
def translate_en_to_ta(text):
    if not isinstance(text, str) or not text.strip():
        return "Please enter English text."

    tokenizer.src_lang = "eng_Latn"
    inputs = tokenizer(
        text.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=400
    ).to(device)

    tamil_token_id = tokenizer.convert_tokens_to_ids("tam_Taml")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            forced_bos_token_id=tamil_token_id,
            max_new_tokens=200,
            num_beams=4
        )

    return tokenizer.batch_decode(output, skip_special_tokens=True)[0]

print(translate_en_to_ta("Hello! How are you?"))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


வணக்கம். எப்படி இருக்கீங்க?


## Try your own text

In [7]:
english = "Nothing is permanent in the world ."
print("English:", english)
print("Tamil:", translate_en_to_ta(english))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


English: Nothing is permanent in the world .
Tamil: இந்த உலகில் எதுவுமே நிரந்தரமானது அல்ல .


## Optional: Gradio interface

In [5]:
import gradio as gr

app = gr.Interface(
    fn=translate_en_to_ta,
    inputs=gr.Textbox(lines=5, label="Enter English text"),
    outputs=gr.Textbox(lines=5, label="Tamil translation"),
    title="English to Tamil Translator",
    description="Translation powered by a pretrained Hugging Face NLLB model."
)
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0947a8bb1c345e4b8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## If it fails
- **CUDA out-of-memory:** In Colab select Runtime → Change runtime type → CPU, or restart the runtime and use a GPU with more memory if available.
- **Download/network error:** Check the internet connection and rerun the model-loading cell.
- **Language token error:** Ensure the tokenizer is initialized with `src_lang="eng_Latn"` and `forced_bos_token_id` uses the ID for `tam_Taml`.
- **Slow first run:** Model weights need to download once; this model is larger than many lightweight translation models.
- Machine translations may contain errors; review the Tamil output before using it in formal work.